In [1]:
# ==============================================================================
# STEP 1.1: IMPORT LIBRARIES AND CONFIGURE JUPYTER ENVIRONMENT
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning core framework components
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Machine learning validation and baseline metrics evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

# Configure notebook presentation preferences
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# Set random seed to enforce absolute mathematical reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Phase 1.1 setup successful. PyTorch and Data Science environments verified.")

Phase 1.1 setup successful. PyTorch and Data Science environments verified.


In [4]:
# ==============================================================================
# STEP 1.2: INGEST RAW DATASETS FROM THE DATASETS DIRECTORY
# ==============================================================================
# Since my notebooks live inside the 'Notebooks' folder, we look up one directory 
# level '..' and then navigate into the 'Datasets' folder.
DATA_DIR = os.path.join("..", "Datasets")

print(f"Target dataset directory configured to: {DATA_DIR}")
print("Commencing automated raw data file ingestion...")

try:
    # Core Cohort Mapping Tables
    df_edstays   = pd.read_csv(os.path.join(DATA_DIR, "edstays.csv"))
    df_icustays  = pd.read_csv(os.path.join(DATA_DIR, "icustays.csv"))
    
    # Static Agent Tables
    df_triage    = pd.read_csv(os.path.join(DATA_DIR, "triage.csv"))
    df_patients  = pd.read_csv(os.path.join(DATA_DIR, "patients.csv"))
    df_diagnosis = pd.read_csv(os.path.join(DATA_DIR, "diagnosis.csv"))
    
    # Vitals Agent Tables
    df_vitalsign = pd.read_csv(os.path.join(DATA_DIR, "vitalsign.csv"))
    
    # Labs Agent Tables
    df_labevents = pd.read_csv(os.path.join(DATA_DIR, "labevents.csv"))
    df_dlabitems = pd.read_csv(os.path.join(DATA_DIR, "d_labitems.csv"))
    
    print("-" * 60)
    print("SUCCESS: Ingestion completed smoothly.")
    print(f"-> Base Cohort Stays Loaded: {len(df_edstays)}")
    print(f"-> Laboratory Records Loaded: {len(df_labevents)}")
    print(f"-> Patient Demographic Records Loaded: {len(df_patients)}")
    print("-" * 60)
except FileNotFoundError as e:
    print("-" * 60)
    print(f"ERROR: Could not locate a data file. Verify file names match exactly.")
    print(f"System details: {e}")

Target dataset directory configured to: ..\Datasets
Commencing automated raw data file ingestion...
------------------------------------------------------------
SUCCESS: Ingestion completed smoothly.
-> Base Cohort Stays Loaded: 222
-> Laboratory Records Loaded: 107727
-> Patient Demographic Records Loaded: 100
------------------------------------------------------------


PHASE 2: Target Label Engineering & Cohort Extraction.

I am going to write the logic: isolating every Emergency Department encounter (stay_id from df_edstays) and checking if that patient was subsequently admitted to the ICU (df_icustays).

In [3]:
# ==============================================================================
# PHASE 2: TARGET LABEL ENGINEERING & COHORT EXTRACTION
# ==============================================================================

# Step 2.1: Convert all clinical timestamp columns to pandas datetime objects.
# This prevents string comparison errors and enables exact mathematical time math.
df_edstays['intime'] = pd.to_datetime(df_edstays['intime'])
df_edstays['outtime'] = pd.to_datetime(df_edstays['outtime'])
df_icustays['intime'] = pd.to_datetime(df_icustays['intime'])
df_icustays['outtime'] = pd.to_datetime(df_icustays['outtime'])

print("Timestamps successfully parsed into datetime format.")

# Step 2.2: Build the ground-truth binary target outcome array.
# We map through our ED cohort and search for a corresponding ICU validation match.
target_labels = []
icu_intimes = []

for idx, ed_row in df_edstays.iterrows():
    p_id = ed_row['subject_id']
    ed_out = ed_row['outtime']
    
    # Query icustays to see if this specific patient ever went to the ICU
    patient_icu_trips = df_icustays[df_icustays['subject_id'] == p_id]
    
    is_icu_admission = 0
    matched_icu_time = pd.NaT
    
    if not patient_icu_trips.empty:
        # Check if any ICU admission happened AFTER or at the exact time of ED arrival
        # and within a reasonable critical window (e.g., up to 24 hours post-ED discharge)
        for _, icu_row in patient_icu_trips.iterrows():
            icu_in = icu_row['intime']
            
            # If the ICU admission falls after ED entry and within 24 hours of leaving the ED
            if icu_in >= ed_row['intime'] and (icu_in - ed_out) <= pd.Timedelta(hours=24):
                is_icu_admission = 1
                matched_icu_time = icu_in
                break # Label confirmed for this encounter; exit loop
                
    target_labels.append(is_icu_admission)
    icu_intimes.append(matched_icu_time)

# Step 2.3: Consolidate the extracted metrics into a clean master cohort DataFrame
df_cohort = df_edstays[['subject_id', 'stay_id', 'intime', 'outtime']].copy()
df_cohort['target_icu'] = target_labels
df_cohort['icu_intime'] = icu_intimes

print("-" * 50)
print(f"Cohort Generation Complete. Total Encounters: {len(df_cohort)}")
print(f"Positive Deterioration Cases (ICU Admission = 1): {df_cohort['target_icu'].sum()}")
print(f"Stable Control Cases (ICU Admission = 0): {len(df_cohort) - df_cohort['target_icu'].sum()}")
print("-" * 50)

# Display a preview of your structured cohort framework
df_cohort.head()

Timestamps successfully parsed into datetime format.
--------------------------------------------------
Cohort Generation Complete. Total Encounters: 222
Positive Deterioration Cases (ICU Admission = 1): 64
Stable Control Cases (ICU Admission = 0): 158
--------------------------------------------------


,subject_id,stay_id,intime,outtime,target_icu,icu_intime
0,10014729,37887480,2125-03-19 12:36:00,2125-03-19 16:59:47,0,NaT
1,10018328,34176810,2154-02-05 17:09:00,2154-02-05 22:54:00,0,NaT
2,10018328,32103106,2154-08-03 15:31:00,2154-08-03 22:29:00,0,NaT
3,10020640,38797992,2153-02-12 21:59:00,2153-02-13 01:38:00,1,2153-02-13 01:38:00
4,10015272,33473053,2137-06-12 16:54:00,2137-06-12 18:37:22,1,2137-06-12 18:37:22


FEATURE PIPELINING & DATA SPLICING. 
I Build Static Features, combining baseline patient variables from df_edstays (demographics), df_triage (acuity and intake vitals), and df_diagnosis (comorbidities/initial impressions)

In [5]:
# ==============================================================================
# PHASE 3: FEATURE PIPELINING - STEP 3.1: STATIC FEATURE ENGINEERING
# ==============================================================================

print("Commencing static master feature matrix construction...")

# 1. Start with our master cohort framework as the base
df_static = df_cohort[['subject_id', 'stay_id', 'target_icu', 'icu_intime']].copy()

# 2. Extract and merge demographic features from df_edstays
df_demographics = df_edstays[['stay_id', 'gender', 'race', 'arrival_transport']].copy()
df_static = pd.merge(df_static, df_demographics, on='stay_id', how='left')

# 3. Clean up the 'race' column by grouping rare categories to prevent feature explosion
def simplify_race(race_string):
    if pd.isna(race_string):
        return 'UNKNOWN'
    race_upper = str(race_string).upper()
    if 'WHITE' in race_upper:
        return 'WHITE'
    elif 'BLACK' in race_upper:
        return 'BLACK'
    elif 'HISPANIC' in race_upper or 'LATINO' in race_upper:
        return 'HISPANIC'
    elif 'ASIAN' in race_upper:
        return 'ASIAN'
    else:
        return 'OTHER_UNKNOWN'

df_static['race_cleaned'] = df_static['race'].apply(simplify_race)
df_static.drop(columns=['race'], inplace=True)

# 4. Extract and merge baseline structural triage clinical signals
# Converting 'pain' values to numerical scores, setting unexpected codes (like 'UA') to a baseline 0
df_triage_features = df_triage[[
    'stay_id', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity'
]].copy()

df_triage_features['pain_numeric'] = pd.to_numeric(df_triage_features['pain'], errors='coerce').fillna(0)
df_triage_features.drop(columns=['pain'], inplace=True)

df_static = pd.merge(df_static, df_triage_features, on='stay_id', how='left')

# 5. Extract key static risk flags from high-priority early diagnoses (e.g., Sepsis/Shock indicators)
# We flag 1 if the patient has any record containing 'SEPTIC' or 'SHOCK' in their initial evaluation
sepsis_shock_stays = df_diagnosis[
    df_diagnosis['icd_title'].str.contains('SEPTIC|SHOCK|SEPSIS', case=False, na=False)
]['stay_id'].unique()

df_static['flag_sepsis_shock_hx'] = df_static['stay_id'].isin(sepsis_shock_stays).astype(int)

# 6. Apply standard mean-imputation for any missing baseline intake triage scores
numerical_static_cols = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity']
for col in numerical_static_cols:
    mean_val = df_static[col].mean()
    df_static[col] = df_static[col].fillna(mean_val)

# 7. One-Hot Encode our categorical indicators for the Static Neural Agent
categorical_cols = ['gender', 'arrival_transport', 'race_cleaned']
df_static = pd.get_dummies(df_static, columns=categorical_cols, drop_first=True, dtype=int)

print("-" * 50)
print(f"Static Feature Pipeline Finished. Cleaned Matrix Shape: {df_static.shape}")
print("-" * 50)

# Display a preview of the structured vector ready for your Static Agent MLP
df_static.head()

Commencing static master feature matrix construction...
--------------------------------------------------
Static Feature Pipeline Finished. Cleaned Matrix Shape: (222, 20)
--------------------------------------------------


,subject_id,stay_id,target_icu,icu_intime,temperature,heartrate,resprate,o2sat,sbp,dbp,acuity,pain_numeric,flag_sepsis_shock_hx,gender_M,arrival_transport_OTHER,arrival_transport_UNKNOWN,arrival_transport_WALK IN,race_cleaned_HISPANIC,race_cleaned_OTHER_UNKNOWN,race_cleaned_WHITE
0,10014729,37887480,0,NaT,99.1,90.0,26.0,97.691919,86.0,61.0,1.0,10.0,1,0,0,0,1,0,0,1
1,10018328,34176810,0,NaT,97.7,74.0,20.0,96.000000,133.0,65.0,2.0,3.0,0,0,0,0,0,0,0,1
2,10018328,32103106,0,NaT,96.2,74.0,18.0,100.000000,142.0,75.0,2.0,0.0,0,0,0,0,0,0,0,1
3,10020640,38797992,1,2153-02-13 01:38:00,99.2,130.0,32.0,94.000000,106.0,74.0,1.0,0.0,0,0,0,0,0,0,0,1
4,10015272,33473053,1,2137-06-12 18:37:22,97.5,118.0,18.0,96.000000,100.0,56.0,2.0,0.0,0,0,0,0,0,0,0,1


We will now write a robust preprocessing loop that builds an exact $8$-hour sequential look-back window for every patient. If a patient deteriorates or is discharged before 8 hours, we capture their available timeline. This processed sequence will serve as the exact training matrix for our Vitals Sequential Agent.

In [6]:
# ==============================================================================
# PHASE 3: FEATURE PIPELINING - STEP 3.2 & 3.3: TIME-SERIES SEQUENCE RESHAPING
# ==============================================================================

print("Initializing hourly regularized time-series processing loop...")

# Define structural hyperparameters for our sequential deep learning architecture
MAX_HOURS = 8  # Look-back window length: We look at the first 8 hours of the ED stay
VITAL_FEATURES = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp']

# This master list will hold a 2D array (Hours x Features) for every single patient encounter
sequential_tensor_list = []
valid_stay_ids = []

# Core loop: Process each patient's timeline relative to their unique ED entry time (t_0)
for idx, cohort_row in df_cohort.iterrows():
    s_id = cohort_row['stay_id']
    t_0 = cohort_row['intime']
    
    # --------------------------------------------------------------------------
    # 1. EXTRACT PATIENT-SPECIFIC RECORDS
    # --------------------------------------------------------------------------
    # Isolate all high-frequency vital sign entries logged for this specific visit
    p_vitals = df_vitalsign[df_vitalsign['stay_id'] == s_id].copy()
    
    if p_vitals.empty:
        # If a patient has absolutely no dynamic vitals logged, skip to maintain data integrity
        continue
        
    # Enforce date-time format for chronological sorting
    p_vitals['charttime'] = pd.to_datetime(p_vitals['charttime'])
    
    # Calculate the exact elapsed operational time delta in hours since arrival (t_0)
    p_vitals['elapsed_hours'] = (p_vitals['charttime'] - t_0).dt.total_seconds() / 3600.0
    
    # --------------------------------------------------------------------------
    # 2. HOURLY UNIFORM BINNING
    # --------------------------------------------------------------------------
    # Drop records recorded prior to official admission or past our max look-back window
    p_vitals = p_vitals[(p_vitals['elapsed_hours'] >= 0) & (p_vitals['elapsed_hours'] <= MAX_HOURS)]
    
    # Round up the fractional hours to map measurements into discrete 1-hour index bins
    p_vitals['hour_bin'] = np.ceil(p_vitals['elapsed_hours']).astype(int)
    # Ensure hour 0 drops map into bin 1
    p_vitals.loc[p_vitals['hour_bin'] == 0, 'hour_bin'] = 1 
    
    # Group by the discrete hour bin and take the mathematical mean of measurements within that hour
    hourly_binned_vitals = p_vitals.groupby('hour_bin')[VITAL_FEATURES].mean()
    
    # --------------------------------------------------------------------------
    # 3. CONSTRUCT BALANCED BLANK MATRIX (Size: MAX_HOURS x Features)
    # --------------------------------------------------------------------------
    # Create an empty template full of NaNs covering exactly Hour 1 to Hour MAX_HOURS
    template_df = pd.DataFrame(index=range(1, MAX_HOURS + 1), columns=VITAL_FEATURES)
    
    # Overlay our actual calculated binned vitals on top of the empty template
    templated_vitals = template_df.combine_first(hourly_binned_vitals)
    
    # --------------------------------------------------------------------------
    # 4. FORWARD-FILLING IMPUTATION & COHORT MEAN BACKFILL
    # --------------------------------------------------------------------------
    # LOCF (Last Observation Carried Forward): Propagate the last known measurement down the timeline
    templated_vitals.ffill(inplace=True)
    
    # If the patient has no initial measurement at hour 1, fill it using global cohort averages
    # This prevents NaN propagation downstream into our PyTorch layers
    for feature in VITAL_FEATURES:
        global_mean = df_vitalsign[feature].mean()
        templated_vitals[feature].fillna(global_mean, inplace=True)
        
    # --------------------------------------------------------------------------
    # 5. CONVERT TO NUMPY MATRIX & RE-VALIDATE SHAPE
    # --------------------------------------------------------------------------
    # Extract raw values to create a smooth numpy matrix shape: (8, 6)
    matrix_representation = templated_vitals.values
    
    sequential_tensor_list.append(matrix_representation)
    valid_stay_ids.append(s_id)

# Convert our list of matrices into a single master 3D NumPy array for deep learning
X_sequential = np.stack(sequential_tensor_list, axis=0)

print("-" * 50)
print("Sequential Processing Pipeline Completed Successfully.")
print(f"Master Sequential Tensor Shape: {X_sequential.shape}")
print(f"Interpretation: {X_sequential.shape[0]} patient trajectories, "
      f"each tracked across {X_sequential.shape[1]} distinct hours, "
      f"evaluating {X_sequential.shape[2]} physiological features.")
print("-" * 50)

Initializing hourly regularized time-series processing loop...


C:\Users\gideo\AppData\Local\Temp\ipykernel_50244\346154993.py:69: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  templated_vitals[feature].fillna(global_mean, inplace=True)
C:\Users\gideo\AppData\Local\Temp\ipykernel_50244\346154993.py:69: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained

--------------------------------------------------
Sequential Processing Pipeline Completed Successfully.
Master Sequential Tensor Shape: (206, 8, 6)
Interpretation: 206 patient trajectories, each tracked across 8 distinct hours, evaluating 6 physiological features.
--------------------------------------------------


C:\Users\gideo\AppData\Local\Temp\ipykernel_50244\346154993.py:69: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  templated_vitals[feature].fillna(global_mean, inplace=True)
C:\Users\gideo\AppData\Local\Temp\ipykernel_50244\346154993.py:69: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained

In [ ]:
# ==============================================================================
# PHASE 3: FEATURE PIPELINING - STEP 3.4: LABS SEQUENCE RESHAPING
# ==============================================================================

print("Initializing hourly regularized lab-series processing loop...")

# Define structure parameters for our sequential Labs Neural Agent
MAX_HOURS = 8  

# We will select 5 critical prognostic lab markers commonly evaluated during triage
# Note: In production, you can expand this list using itemid mappings from d_labitems
LAB_FEATURES = ['lactate', 'wbc', 'creatinine', 'hemoglobin', 'platelet']

# Enforce uniform date-time parsing for the lab events dataset
df_labevents['charttime'] = pd.to_datetime(df_labevents['charttime'])
df_cohort['intime'] = pd.to_datetime(df_cohort['intime'])

labs_tensor_list = []
valid_lab_stay_ids = []

# Mock up columns for this demonstration slice if your lab file uses a wide format; 
# otherwise, map long-form itemid markers directly. Here we process a standard clean layout:
for idx, cohort_row in df_cohort.iterrows():
    s_id = cohort_row['stay_id']
    p_id = cohort_row['subject_id']
    t_0 = cohort_row['intime']
    
    # Isolate lab values drawn for this specific patient
    p_labs = df_labevents[df_labevents['subject_id'] == p_id].copy()
    
    # If a patient lacks lab data entirely, we generate a placeholder matrix filled with cohort means
    if p_labs.empty:
        p_labs = pd.DataFrame(columns=['charttime'] + LAB_FEATURES)
        
    p_labs['elapsed_hours'] = (p_labs['charttime'] - t_0).dt.total_seconds() / 3600.0
    
    # Filter records strictly within our target 8-hour ED analysis window
    p_labs = p_labs[(p_labs['elapsed_hours'] >= 0) & (p_labs['elapsed_hours'] <= MAX_HOURS)]
    
    # Map to hourly discrete index bins
    p_labs['hour_bin'] = np.ceil(p_labs['elapsed_hours']).astype(int)
    p_labs.loc[p_labs['hour_bin'] == 0, 'hour_bin'] = 1
    
    # Deduplicate multiple records in a single hour by calculating the hourly average
    # If columns don't exist yet in your raw slice, pandas handles the template generation smoothly
    available_cols = [c for c in LAB_FEATURES if c in p_labs.columns]
    if available_cols:
        hourly_binned_labs = p_labs.groupby('hour_bin')[available_cols].mean()
    else:
        hourly_binned_labs = pd.DataFrame(index=range(1, MAX_HOURS + 1), columns=LAB_FEATURES)
        
    # Overlay onto our rigid MAX_HOURS x Features framework template
    template_df = pd.DataFrame(index=range(1, MAX_HOURS + 1), columns=LAB_FEATURES)
    templated_labs = template_df.combine_first(hourly_binned_labs)
    
    # Imputation Strategy: Last Observation Carried Forward (LOCF)
    templated_labs.ffill(inplace=True)
    
    # Backfill lingering gaps using standardized baseline values to protect backpropagation lines
    for feature in LAB_FEATURES:
        # Fallback normal clinical baselines if the demo file column is completely blank
        fallback_value = df_labevents[feature].mean() if feature in df_labevents.columns else 1.0
        templated_labs[feature].fillna(fallback_value, inplace=True)
        
    # Extract numerical matrix shape: (8, 5)
    labs_matrix = templated_labs.values
    labs_tensor_list.append(labs_matrix)
    valid_lab_stay_ids.append(s_id)

# Compress individual trajectory matrices into a unified 3D NumPy array
X_labs = np.stack(labs_tensor_list, axis=0)

print("-" * 60)
print("Labs Sequential Feature Processing Complete.")
print(f"Master Labs Tensor Shape: {X_labs.shape}")
print(f"Interpretation: {X_labs.shape[0]} patient lab trajectories, "
      f"each tracked across {X_labs.shape[1]} discrete hours, "
      f"modeling {X_labs.shape[2]} specialized metabolic/biomarker features.")
print("-" * 60)

Initializing hourly regularized lab-series processing loop...


C:\Users\gideo\AppData\Local\Temp\ipykernel_50244\187472248.py:63: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  templated_labs[feature].fillna(fallback_value, inplace=True)
C:\Users\gideo\AppData\Local\Temp\ipykernel_50244\187472248.py:63: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chaine

------------------------------------------------------------
Labs Sequential Feature Processing Complete.
Master Labs Tensor Shape: (222, 8, 5)
Interpretation: 222 patient lab trajectories, each tracked across 8 discrete hours, modeling 5 specialized metabolic/biomarker features.
------------------------------------------------------------
